![Mi imagen](https://drive.google.com/uc?id=1I5sS373_3fgoQd-UCy5L-5tlkMSelL0T)

# **Transformada de Laplace**

Si $f(t)$ es una función definida en $[0,\infty[$ con $t$ y $f$ reales, entonces la transformada de Laplace de la función $f$ se denota por $\mathcal{L}\lbrace f(t) \rbrace = F(s)$ y se define como la integral
$$ \mathcal{L}\lbrace f(t) \rbrace = F(s) = \int_{0}^{\infty} f(t) e^{-st} dt $$
siempre que la anterior sea convergente.

### **Propiedades de las transformadas de Laplace**
Las Transformadas de Laplace poseen algunas propiedades que también nos van a facilitar el trabajo de resolverlas, algunas de ellas son:

* La Transformada de Laplace es un operador lineal: Esta propiedad nos dice la Transformada de Laplace de una suma, es igual a la suma de las Transformadas de Laplace de cada uno de los términos. Es decir:

$$\mathcal{L}\lbrace c_1f_1(t)+c_2f_2(t)\rbrace=c_1\mathcal{L}\lbrace f_1(t)\rbrace+c_2\mathcal{L}\lbrace f_2(t)\rbrace.$$

* La Transformada de Laplace de la primera derivada: Esta propiedad nos dice que si $f(t)$ es continua y $f'(t)$ es continua en el intervalo $ 0 \leq t \leq \alpha $. Entonces la Transformada de Laplace de la primera derivada es:
$$\mathcal{L}\lbrace f'(t)\rbrace =s\mathcal{L}\lbrace f(t)\rbrace−f(0).$$

* La Transformada de Laplace de derivadas de orden superior: Esta propiedad es la generalización de la propiedad anterior para derivadas de orden $n$. Su formula es:

$$\mathcal{L}\lbrace f^{(n)}(t)\lbrace=s^n\mathcal{L}\lbrace f(t)\lbrace −s^{n−1}f(0)-\ldots−f^{(n−1)}(0)= s^n\mathcal{L}\lbrace f(t)\rbrace −\sum_{i=1}^n s^{n−i}f^{(i−1)}(0)$$

### **Resolución de ecuaciones diferenciales con transformada de Laplace**
La principal ventaja de utilizar Transformadas de Laplace es que transforma la Ecuación diferencial en una ecuación algebraica, lo que simplifica el proceso de cálculo su solución. Una vez obtenida esta última, se revierte la transformación para obtener la solución que se buscaba de la ecuaciión diferencial.

El encontrar las transformaciones iniciales de los términos de la ecuación diferencial que queramos resolver y luego obtener las transformaciones inversas finales para tener la solución son los procesos más laboriosos. Las operaciones intermedias son más simples.

El software `Python` y su biblioteca `SymPy` ofrecen las herramientas que ayudan a resolver ecuaciones diferenciales utilizando la Transformada de Laplace.

---

##**1. Importar SymPy y definir símbolos**

####**Explicación**

* $t$ se define como real y positiva porque en la transformada de Laplace trabajamos con $\geq0$.

* $s$ es compleja, aunque SymPy la tratará simbólicamente.

* $y$ es la función solución que buscamos, dependiente de $t$.

In [1]:
import sympy as sp

# Definir la variable independiente (tiempo) y la variable de Laplace
t = sp.symbols('t', real=True, positive=True)  # t >= 0
s = sp.symbols('s', complex=True)              # variable compleja de Laplace

# Definir la función incógnita (dependiente del tiempo)
y = sp.Function('y')(t)   # o bien: f = sp.Function('f'); luego usas f(t)

##**2. Escribir la ecuación diferencial**

Resolver la siguiente ecuación diferencial:

$$ y''+3y'+2y=0$$

con las condiciones iniciales: $y(0)=2$ y $y'(0)=-3$.


In [6]:
# Definir la EDO (miembro izquierdo = 0)
edo = y.diff(t, 2) + 3*y.diff(t) + 2*y
edo

2*y(t) + 3*Derivative(y(t), t) + Derivative(y(t), (t, 2))

In [7]:
# La ecuación es edo == 0
sp.Eq(edo, 0)

Eq(2*y(t) + 3*Derivative(y(t), t) + Derivative(y(t), (t, 2)), 0)

##**3. Aplicar la transformada de Laplace a toda la ecuación**
La documentación utiliza una función auxiliar `_laplace_apply` (interna de SymPy) para reemplazar `LaplaceTransform(f(t).diff(t,...), t, s)` por la expresión algebraica en $s$ y las condiciones iniciales.

En versiones recientes de SymPy ($\geq 1.9$) puedes usar directamente `sp.laplace_transform con doit()` o usar `sp.integrate` con la definición.

La forma más automática (recomendada):
SymPy tiene la función `laplace_transform` que ya sabe expandir derivadas si le das la opción `doit()`:

In [9]:
# Obtener una expresión con LaplaceTransform de f(t) y sin(t)
L_expr = sp.laplace_transform(edo, t, s, noconds=True)
display(L_expr)

# Sustituir manualmente las transformadas de las derivadas usando la propiedad:
# L{f'(t)} = s*F(s) - f(0)
# L{f''(t)} = s^2*F(s) - s*f(0) - f'(0)

# Definir F(s) como la transformada de f(t)
F = sp.Function('F')(s)

# Crear un diccionario de reemplazo
reemplazo = {
    sp.laplace_transform(y, t, s, noconds=True): F,
    sp.laplace_transform(y.diff(t), t, s, noconds=True): s*F - y.subs(t, 0),
    sp.laplace_transform(y.diff(t, t), t, s, noconds=True): s**2*F - s*y.subs(t, 0) - y.diff(t).subs(t, 0),
    sp.laplace_transform(sp.sin(t), t, s, noconds=True): 1/(s**2 + 1)
}

# Aplicar el reemplazo
L_edo_expandida = L_expr.subs(reemplazo)
L_edo_expandida


s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - s*y(0) + 2*LaplaceTransform(y(t), t, s) - 3*y(0) - Subs(Derivative(y(t), t), t, 0)

s**2*F(s) + 3*s*F(s) - s*y(0) + 2*F(s) - 3*y(0) - Subs(Derivative(y(t), t), t, 0)

In [10]:
L_edo_expandida = sp.laplace_transform(edo, t, s, noconds=True).doit()
L_edo_expandida

s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - s*y(0) + 2*LaplaceTransform(y(t), t, s) - 3*y(0) - Subs(Derivative(y(t), t), t, 0)

Esto evalúa directamente las transformadas de derivadas usando condiciones iniciales nulas por defecto.

Para introducir condiciones iniciales concretas, hay que usar `subs` después.



##**4. Introducir las condiciones iniciales**
Suponiendo $y(0)=2$ e $y'(0)=-3$:

In [11]:
# Sustituir numéricamente (o simbólicamente) los valores iniciales
L_edo_con_ci = L_edo_expandida.subs({y.subs(t, 0): 2, y.diff(t).subs(t, 0): -3})
L_edo_con_ci

s**2*LaplaceTransform(y(t), t, s) + 3*s*LaplaceTransform(y(t), t, s) - 2*s + 2*LaplaceTransform(y(t), t, s) - 3

##**5. Resolver la ecuación algebraica para $F(s)$**
La ecuación transformada es `L_edo_con_ci == 0`. Despejamos $F(s)$:

In [12]:
'''# Resolver para F(s) (la transformada de la solución)
sol_F = sp.solve(L_edo_con_ci, F)
F_s = sol_F[0]   # primera solución (única)
'''

from sympy import LaplaceTransform

# Definir F(s) como la transformada de f(t)
F = sp.Function('F')(s)

# Sustituir LaplaceTransform(f(t), t, s) por F(s) en la ecuación
# 'f' itself represents f(t) because it was defined as f = sp.Function('f')(t)
L_edo_final = L_edo_con_ci.subs(LaplaceTransform(y, t, s), F)

# Resolver para F(s) (la transformada de la solución)
sol_F = sp.solve(L_edo_final, F)
F_s = sol_F[0]   # primera solución (única)
F_s

(2*s + 3)/(s**2 + 3*s + 2)

##**6. Aplicar la transformada inversa de Laplace**

In [13]:
# Obtener la solución en el tiempo
sol_t = sp.inverse_laplace_transform(F_s, s, t)
sol_t

exp(-t) + exp(-2*t)

Por defecto, inverse_laplace_transform incluye un escalón unitario $\text{Heaviside}(t)$. Como trabajamos con $t>0$, se puede simplificar.

##**7. Mostrar resultado y verificar**

In [14]:
# Simplificar (opcional)
sol_t_simplificada = sp.simplify(sol_t)
print("Solución de la EDO")
sp.Eq(y, sol_t_simplificada)


Solución de la EDO


Eq(y(t), (exp(t) + 1)*exp(-2*t))

##**8. Verificar el resultado y verificar**

In [15]:
# Verificar que las condiciones iniiciales satiisfacen la EDO y substituyéndolas
solucion_t = sp.simplify(edo.subs(y, sol_t_simplificada))
print("EDO verificada:", sp.Eq(solucion_t, 0))
print("y(0) =", sol_t_simplificada.subs(t, 0))
print("y'(0) =", sol_t_simplificada.diff(t).subs(t, 0))

EDO verificada: True
y(0) = 2
y'(0) = -3


In [16]:
# Verificación con la función checkodesol()
# checkodesol() devuelve una tupla: (es_solución_EDO, es_solución_EDO_simplificada)

# Si la EDO es de segundo orden, también comprueba las condiciones iniciales cuando se proporcionan
edo_bien_resuelta = sp.checkodesol(edo, sol_t_simplificada, y)[0]
if edo_bien_resuelta:
    print("La solución hallada resuelve correctamente la EDO")
else:
    print("La solución hallada NO resuelve correctamente la EDO")

# Verificación de las condiciones iniciales explícitamente (ya se hizo, pero para mantener la consistencia en esta parte)
print("y(0) =", sol_t_simplificada.subs(t, 0))
print("y'(0) =", sol_t_simplificada.diff(t).subs(t, 0))

La solución hallada resuelve correctamente la EDO
y(0) = 2
y'(0) = -3


##**Ejercicios**

### Ejercicio 01

Usando transformada de la Laplace, resolver:

$$ y''+y+1=e^{t}\sin(2t) \quad ; y(0)=1 ; y'(0)=2$$

In [25]:
y = sp.Function('y')(t)

edo = sp.Eq(y.diff(t, 2) + y + 1, sp.exp(t) * sp.sin(2*t))
L_edo = sp.laplace_transform(edo.lhs - edo.rhs, t, s, noconds=True)

sol = sp.solve(L_edo, sp.laplace_transform(y, t, s, noconds=True))[0].subs({y.subs(t, 0): 1, y.diff(t).subs(t, 0): 2})
sol = sp.apart(sol)

sol_t = sp.inverse_laplace_transform(sol, s, t)
sol_t

-exp(t)*sin(2*t)/10 - exp(t)*cos(2*t)/5 + 12*sin(t)/5 + 11*cos(t)/5 - 1

### Ejercicio 02

Usando transformada de la Laplace, resolver:


$$ y''+y=\frac{1}{\sqrt{t}} + \sqrt{t}  \quad ; y(0)=1 ; y'(0)=2$$

In [29]:
y = sp.Function('y')(t)

edo = sp.Eq(y.diff(t, 2) + y, 1/sp.sqrt(t) + sp.sqrt(t))
L_edo = sp.laplace_transform(edo.lhs - edo.rhs, t, s, noconds=True)

sol = sp.solve(L_edo, sp.laplace_transform(y, t, s, noconds=True))[0].subs({y.subs(t, 0): 1, y.diff(t).subs(t, 0): 2})
sol = sp.expand(sol)

sol_t = sp.inverse_laplace_transform(sol, s, t)
sol_t

sqrt(pi)*InverseLaplaceTransform(sqrt(s)/(2*s**4 + 2*s**2), s, t, _None) + sqrt(pi)*InverseLaplaceTransform(s**(3/2)/(s**4 + s**2), s, t, _None) + 2*sin(t) + cos(t)

### Ejercicio 03

Usando transformada de la Laplace, resolver:

$$ ty''+y'+y = 0$$

In [35]:
y = sp.Function('y')(t)

edo = sp.Eq(t * y.diff(t, 2) + y.diff(t) + y, 0)
L_edo = sp.laplace_transform(edo.lhs - edo.rhs, t, s, noconds=True)
display(L_edo)

sol = sp.solve(L_edo, sp.laplace_transform(y, t, s, noconds=True))[0]
sol

s*LaplaceTransform(y(t), t, s) + LaplaceTransform(y(t), t, s) - y(0) - Derivative(s**2*LaplaceTransform(y(t), t, s) - s*y(0) - Subs(Derivative(y(t), t), t, 0), s)

(y(0) + Derivative(s**2*LaplaceTransform(y(t), t, s) - s*y(0) - Subs(Derivative(y(t), t), t, 0), s))/(s + 1)

### Ejercicio 04:

* a) Con la ayuda de la librería `sympy`, determine $\mathcal{L}\left\lbrace \dfrac{1}{\sqrt{t}} \right\rbrace$. A partir de su resultado deduzca $\mathcal{L}^{-1}\left\lbrace \dfrac{1}{\sqrt{s}} \right\rbrace$

* b) Usando el resultado anterior, resolver la ecuación integral

$$ C = \int_{0}^{t} \frac{f(t)}{\sqrt{t-\tau}} d\tau $$

para $C\in \mathbb{R}$ y $f$ función incógnita.

Recordar que
$$(f * g)(t) =  \int_{0}^{t} f(t)g(t-\tau) d\tau $$
y
$$ \mathcal{L}\lbrace f * g \rbrace = \mathcal{L}\lbrace f \rbrace \mathcal{L}\lbrace g \rbrace $$


In [36]:
# 1. Definimos las variables simbólicas
t = sp.symbols('t', positive=True)
s = sp.symbols('s', positive=True)

# 2. Definimos la función
f = 1 / sp.sqrt(t)

# 3. Calculamos la Transformada de Laplace
# noconds=True simplifica la salida ocultando las condiciones de convergencia
F_s = sp.laplace_transform(f, t, s, noconds=True)

print("La transformada de Laplace de 1/sqrt(t) es:")
sp.pprint(F_s)

La transformada de Laplace de 1/sqrt(t) es:
√π
──
√s


### Ejercicio 05

La función error de aparece en teoría de probabilidad y se define como

$$erf(t)=\frac{2}{\sqrt{\pi}}\int_{0}^{t}e^{-x^2}dx$$

esta función está incorporada en la librería `sympy` definida por `sp.erf(t)` (definiendo previamente el simbolo `t`)

* a) Usando el comando `checkodesol` verifique que $y=c_1 erf(t)+c_2$ es solución general de la EDO homogénea

$$y'' + 2t y'=0$$

* b) Calcule $\mathcal{L}\lbrace erf(\sqrt{t}) \rbrace$.

* c) Sin utilizar el comando `dsol`, pero con ayuda de `sympy`, resuelva el PVI

$$y' = erf(t) + t \quad , \quad y(0)=-1$$

In [37]:
from IPython.display import display

# Definición de variables simbólicas
t = sp.symbols('t', real=True)
s = sp.symbols('s')
c1, c2, C = sp.symbols('c1 c2 C')

print("--- PARTE A: Verificación de la EDO homogénea ---")
# 1. Definimos la función y(t) genérica para plantear la EDO
y = sp.Function('y')(t)

# 2. Planteamos la ecuación diferencial: y'' + 2ty' = 0
edo = sp.Eq(y.diff(t, 2) + 2 * t * y.diff(t), 0)

# 3. Definimos la solución propuesta: y = c1*erf(t) + c2
sol_propuesta = sp.Eq(y, c1 * sp.erf(t) + c2)

# 4. Usamos checkodesol para verificar si es solución
verificacion = sp.checkodesol(edo, sol_propuesta)

# checkodesol devuelve una tupla (True, 0) si es solución correcta
if verificacion[0]:
    print("¡Verificado! La propuesta ES solución de la EDO.\n")
else:
    print("La propuesta NO es solución.\n")


print("--- PARTE B: Transformada de Laplace de erf(sqrt(t)) ---")
# 1. Definimos la función
f_t = sp.erf(sp.sqrt(t))

# 2. Calculamos la transformada de Laplace (noconds=True evita que muestre las condiciones de convergencia)
F_s = sp.laplace_transform(f_t, t, s, noconds=True)

print("La Transformada de Laplace L{erf(sqrt(t))} es:")
display(F_s)
print("\n")


print("--- PARTE C: Resolver el PVI sin usar dsolve ---")
# La ecuación es y' = erf(t) + t. 
# Como es una derivada directa que depende solo de t, resolvemos integrando directamente.

# 1. Definimos la derivada
dy_dt = sp.erf(t) + t

# 2. Integramos con respecto a t (esto nos da la función original y(t) sin la constante)
y_int = sp.integrate(dy_dt, t)

# Planteamos que y(t) = y_int + C
y_general = y_int + C

# 3. Aplicamos la condición inicial: y(0) = -1
# Evaluamos la solución general en t = 0 y la igualamos a -1
eq_condicion_inicial = sp.Eq(y_general.subs(t, 0), -1)

# 4. Resolvemos para la constante C
valor_C = sp.solve(eq_condicion_inicial, C)[0]

# 5. Sustituimos el valor de C en la solución general para obtener la particular
solucion_PVI = y_general.subs(C, valor_C)

print("La solución particular al PVI y(0) = -1 es: y(t) =")
display(solucion_PVI)

--- PARTE A: Verificación de la EDO homogénea ---
¡Verificado! La propuesta ES solución de la EDO.

--- PARTE B: Transformada de Laplace de erf(sqrt(t)) ---
La Transformada de Laplace L{erf(sqrt(t))} es:


1/(s*sqrt(s + 1))



--- PARTE C: Resolver el PVI sin usar dsolve ---
La solución particular al PVI y(0) = -1 es: y(t) =


t**2/2 + t*erf(t) - 1 - 1/sqrt(pi) + exp(-t**2)/sqrt(pi)

### Ejercicio 06:

Se sabe que, para $f(t)=t^n$ con $n\in \mathbb{N}$

$$\mathcal{L}\lbrace t^n \rbrace = \frac{n!}{s^{n+1}}$$

sin embargo, esta fórmula deja de ser cierta si $n \in \mathbb{R}$.

* a) Usando librería SymPy, calcular $\mathcal{L}\lbrace t^{\frac{p}{2}} \rbrace$ para $p=1,3,5,7$. Sugerencia: Se recomienda utilizar `sp.Rational(a,b)` para representar números racionales de la forma $\frac{a}{b}$.

* b) A partir de los resultados de a) deduciir una fórmula para $\mathcal{L}\lbrace t^{\frac{2n-1}{2}} \rbrace$ para $ n \in \mathbb{N}$

* c) Usando estos resultados, resolver el PVI

$$y' = \sqrt{t} \quad ; \quad y(0)=1$$

In [ ]:
# 1. Definimos las variables simbólicas (asumiendo reales positivos para Laplace)
t = sp.symbols('t', positive=True)
s = sp.symbols('s', positive=True)

print("--- PARTE A: Transformadas de Laplace ---")
# 2. Iteramos sobre los valores de p
valores_p = [1, 3, 5, 7]

for p in valores_p:
    # Definimos la función usando sp.Rational para el exponente fraccionario
    f = t**(sp.Rational(p, 2))
    
    # Calculamos la Transformada de Laplace
    F_s = sp.laplace_transform(f, t, s, noconds=True)
    
    print(f"Para p={p}, f(t) = t^({p}/2) :")
    display(F_s)
    print("-" * 40)

Parte b) Deducción de la fórmula generalQueremos encontrar una fórmula general para $\mathcal{L}\left\lbrace t^{\frac{2n-1}{2}} \right\rbrace$ donde $n \in \mathbb{N}$.Observemos cómo se relacionan los valores de $n$ con los resultados de la parte a) (donde el exponente $\frac{p}{2}$ equivale a $\frac{2n-1}{2}$):Si $n=1 \implies \frac{2(1)-1}{2} = \frac{1}{2}$. Resultado: $\dfrac{1 \cdot \sqrt{\pi}}{2^1 \cdot s^{3/2}}$Si $n=2 \implies \frac{2(2)-1}{2} = \frac{3}{2}$. Resultado: $\dfrac{(1 \cdot 3) \cdot \sqrt{\pi}}{2^2 \cdot s^{5/2}}$Si $n=3 \implies \frac{2(3)-1}{2} = \frac{5}{2}$. Resultado: $\dfrac{(1 \cdot 3 \cdot 5) \cdot \sqrt{\pi}}{2^3 \cdot s^{7/2}}$Si $n=4 \implies \frac{2(4)-1}{2} = \frac{7}{2}$. Resultado: $\dfrac{(1 \cdot 3 \cdot 5 \cdot 7) \cdot \sqrt{\pi}}{2^4 \cdot s^{9/2}}$Análisis de patrones:Numerador: Es el producto de los primeros números impares hasta $2n-1$. A esto se le conoce matemáticamente como Doble Factorial y se denota como $(2n-1)!!$. Todo esto va multiplicado por $\sqrt{\pi}$.Denominador (constante): Las potencias de $2$ crecen de acuerdo a $n$: $2^1, 2^2, 2^3, 2^4$. Es decir, es $2^n$.Denominador (variable $s$): El exponente de $s$ siempre es una unidad mayor que el exponente original de $t$. Es decir, $\frac{2n-1}{2} + 1 = \frac{2n+1}{2}$.Por lo tanto, la fórmula deducida es:$$\mathcal{L}\left\lbrace t^{\frac{2n-1}{2}} \right\rbrace = \frac{(2n-1)!! \sqrt{\pi}}{2^n s^{\frac{2n+1}{2}}}$$(Nota: $(2n-1)!! = 1 \cdot 3 \cdot 5 \cdots (2n-1)$)Parte c) Resolución del PVITenemos el problema de valor inicial:$$y' = \sqrt{t} \quad ; \quad y(0)=1$$Lo reescribimos como:$$y' = t^{1/2}$$Paso 1: Aplicar la Transformada de LaplaceAplicamos $\mathcal{L}$ a ambos lados de la EDO:$$\mathcal{L}\{y'\} = \mathcal{L}\{t^{1/2}\}$$Usando las propiedades de la derivada y nuestro resultado de la parte a) para $p=1$ ($n=1$):$$sY(s) - y(0) = \frac{\sqrt{\pi}}{2 s^{3/2}}$$Paso 2: Sustituir la condición inicial y despejar $Y(s)$Sustituimos $y(0) = 1$:$$sY(s) - 1 = \frac{\sqrt{\pi}}{2 s^{3/2}}$$$$sY(s) = 1 + \frac{\sqrt{\pi}}{2 s^{3/2}}$$Dividimos todo entre $s$:$$Y(s) = \frac{1}{s} + \frac{\sqrt{\pi}}{2 s^{5/2}}$$Paso 3: Aplicar la Transformada Inversa ($\mathcal{L}^{-1}$)Aplicamos la inversa a cada término por separado:$$y(t) = \mathcal{L}^{-1}\left\lbrace \frac{1}{s} \right\rbrace + \mathcal{L}^{-1}\left\lbrace \frac{\sqrt{\pi}}{2 s^{5/2}} \right\rbrace$$El primer término es directo: $\mathcal{L}^{-1}\left\lbrace \frac{1}{s} \right\rbrace = 1$.Para el segundo término, miramos los resultados de nuestra parte a). Sabemos que para $p=3$ ($n=2$):$$\mathcal{L}\left\lbrace t^{3/2} \right\rbrace = \frac{3\sqrt{\pi}}{4 s^{5/2}}$$Nosotros necesitamos invertir $\dfrac{\sqrt{\pi}}{2 s^{5/2}}$. Podemos manipular nuestra fracción conocida algebraicamente:$$\frac{3\sqrt{\pi}}{4 s^{5/2}} = \frac{3}{2} \cdot \left( \frac{\sqrt{\pi}}{2 s^{5/2}} \right)$$Despejando lo que está entre paréntesis:$$\frac{\sqrt{\pi}}{2 s^{5/2}} = \frac{2}{3} \mathcal{L}\left\lbrace t^{3/2} \right\rbrace$$Por lo tanto, la transformada inversa de ese bloque es $\dfrac{2}{3}t^{3/2}$.Solución Final:Uniendo los resultados, la solución al problema de valor inicial es:$$y(t) = 1 + \frac{2}{3}t^{3/2}$$